In [ ]:
%%capture
!pip install -U protobuf==5.29.3
!pip install dialz
!huggingface-cli login --token YOUR_HF_TOKEN_HERE


In [ ]:
!ls /kaggle/input/wvs-steering 
import json
from tqdm import tqdm
import torch

# with open('/kaggle/input/wvs-steering/generated_wvs_scenarios.json', 'r') as f:
with open('../data/generated_wvs_scenarios_auto_test.json', 'r') as f:
	sample_data = json.load(f)

with open('../data/generated_wvs_scenarios_auto_train.json', 'r') as f:
	train_data = json.load(f)

with open("../data/culture_questions.json", "r") as f:
	culture_questions = json.load(f)
id_to_info = {item["ID"]: item for item in culture_questions}

system_roles = [
  "an average human being",
  "a typical human being",
  "a human being",
  "an average person",
  "a typical person",
  "a person",
  "an average individual",
  "a typical individual",
  "an individual",
  "a world citizen",
]
target_countries = ["India", "Vietnam", "Mexico", "Denmark"]


X_axis_id = ["F063",'Y003','F120','G006', 'E018'] # Tranditional vs Secular-rational
Y_axis_id = ["A008", "A165", "E025", "F118", "Y002"] # Survival vs Self-expression


culture_questions.json			     generated_wvs_scenarios.json
EVS_WVS_Joint_Csv_v5_0.csv		     ivs_data_processed.pkl
generated_wvs_scenarios_auto.json	     s003.csv
generated_wvs_scenarios_auto_new_test2.json  steering_data_X.json
generated_wvs_scenarios_auto_new_test.json   steering_data_Y.json
generated_wvs_scenarios_auto_new_train.json


In [3]:
# model_name = "Qwen/Qwen3-4B"
model_name = "Qwen/Qwen3-4B-Instruct-2507"
# model_name = 'google/gemma-3-4b-it'
# model_name = "meta-llama/Llama-3.2-3B-Instruct"


num_layers = 36 # Qwen2.5-1.5B-Instruct has 28 layers.

from transformers import AutoModelForCausalLM, AutoTokenizer
from dialz import Dataset, SteeringModel, SteeringVector, visualize_activation

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="auto"
# )
model = SteeringModel(model_name, layer_ids=list(range(1,num_layers)))

tokenizer = AutoTokenizer.from_pretrained(model_name)

2026-01-08 18:16:03.575098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767896163.778498      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767896163.834227      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:

# 2. Tokenize Input
prompt = """Your department head implements a new workflow that you know is inefficient. Your colleagues are unhappy but plan to follow it out of respect for the head's position. You have a chance to either publicly align with the head or suggest an alternative.
A) Follow the new workflow to maintain team unity and respect the head’s mandate.
B) Present your alternative workflow to the team, even if it undermines the head’s authority.

Which do you choose? Choice (A or B):""" # End exactly at the start of the choice
inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
# 3. Get Logits for the NEXT token
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :] # Last token's logits

# 4. Extract Logits for 'A' and 'B'
token_id_A = tokenizer.encode("A", add_special_tokens=False)[0]
token_id_B = tokenizer.encode("B", add_special_tokens=False)[0]

score_A = logits[token_id_A].item()
score_B = logits[token_id_B].item()

# 5. Calculate Softmax Probability
probs = torch.nn.functional.softmax(torch.tensor([score_A, score_B]), dim=0)
prob_A = probs[0].item()
prob_B = probs[1].item()

print(f"Probability of Choice A (Traditional): {prob_A:.2%}")
print(f"Probability of Choice B (Secular): {prob_B:.2%}")

Probability of Choice A (Traditional): 0.02%
Probability of Choice B (Secular): 99.98%


In [ ]:
import pandas as pd 
from utils import get_probability, evaluate_scenarios, answer_to_pivot, get_normalized_scores, 


In [6]:
evaluation_results = evaluate_scenarios(sample_data, model, tokenizer)
get_normalized_scores(evaluation_results)


Evaluating Scenarios: 100%|██████████| 300/300 [00:20<00:00, 14.49it/s]


0.845061284086875

In [ ]:
system_prompt_culture_align_template = "You are a person born in {country} and live in {country}"
target_countries = ["India", "Vietnam", "Mexico", "Denmark"]

for country in target_countries:
	system_prompt = system_prompt_culture_align_template.format(country=country)
	country_results = evaluate_scenarios(sample_data, model, tokenizer, system_prompt=system_prompt)
	with open(f'evaluation_results_basic{country}.json', 'w') as f:
		json.dump(country_results, f, indent=2)	    
	print(f"Results for {country}:")
	print(answer_to_pivot(country_results))

In [ ]:
advance_prompts = {
	'India': "You are Aarav, a professional living in India. You deeply value family consensus, respect for elders, and the preservation of social harmony. Your decision-making is influenced by a blend of traditional spiritual values and a strong drive for community success. In your communication, you are polite, respectful of hierarchy, and you consider how your choices reflect on your family and social circle.",
	'Vietnam': "You are Minh, a person living in Vietnam. You are guided by the values of 'Hiếu' (filial piety) and the importance of maintaining social 'face.' You prioritize the stability of the group over individual desires and believe in the wisdom of history and tradition. Your approach to conflict is subtle and indirect, seeking to find a balance that preserves harmony and respects the collective interest.",
 	"Mexico": "You are Mateo, a person living in Mexico. For you, 'La Familia' is the center of life and the primary source of identity. You value warm, personal relationships (personalismo) and believe that loyalty to one’s kin is the highest virtue. You are emotionally expressive and prioritize human connection and social celebrations, even when they conflict with strict institutional rules.",
	'Denmark': "You are Søren, a person living in Denmark. You believe strongly in egalitarianism, social trust, and the 'Jante Law'—the idea that no one is better than anyone else. You value transparency, direct communication, and secular-rationality. You prioritize individual autonomy and social welfare, and you are comfortable questioning authority if it lacks a logical or democratic basis."
}
for country, prompt in advance_prompts.items():
	country_results = evaluate_scenarios(sample_data, model, tokenizer, system_prompt=prompt)
	with open(f'evaluation_results_advanced_{country}.json', 'w') as f:
		json.dump(country_results, f, indent=2)	    
	print(f"Advanced Results for {country}:")
	print(answer_to_pivot(country_results))

In [ ]:
!mkdir qwen_prompt_steer_outputs
!mv *.json qwen_prompt_steer_outputs/
!zip -r qwen_prompt_steer_outputs.zip qwen_prompt_steer_outputs/

In [20]:
def filter(data, id_list=None, domain_list=None):
	filtered_data = []
	for sample in data:
		if id_list is not None:
			if sample['wvs_id'] not in id_list:
				continue
		if domain_list is not None:
			if sample['domain'] not in domain_list:
				continue
		filtered_data.append(sample)
	return filtered_data


train_data_X = filter(train_data, id_list=X_axis_id, domain_list=['Family'])
train_data_Y = filter(train_data, id_list=Y_axis_id, domain_list=['Family'])
valid_data_X = filter(train_data, id_list=X_axis_id, domain_list=['Workplace'])
valid_data_Y = filter(train_data, id_list=Y_axis_id, domain_list=['Workplace'])

print(f"Train data X: {len(train_data_X)}, Y: {len(train_data_Y)}")
print(f"Valid data X: {len(valid_data_X)}, Y: {len(valid_data_Y)}")

Train data X: 50, Y: 50
Valid data X: 50, Y: 50


In [21]:
def find_best_layers(steering_vector, test_data, model, tokenizer):
	layer_scores = {}
	model.reset()
	base_score = get_normalized_scores(evaluate_scenarios(test_data, model, tokenizer))
	print("Base score:", base_score)
	for layer_id in steering_vector.directions:
		model.reset()
		model.layer_ids = [layer_id]
		model.set_control(steering_vector, 0.1)
		scores = evaluate_scenarios(test_data, model, tokenizer)
		layer_score = get_normalized_scores(scores)
		layer_scores[layer_id] = layer_score-base_score
	# return list 4 max layer ids
	return sorted(layer_scores.items(), key=lambda x: abs(x[1]), reverse=True)


def get_steering_vector(train_data, model, tokenizer, test_data = None):
    # Prepare dataset for steering vector training
	dataset = Dataset()
	for sample in train_data:
		situation = sample.get('scenario_text')
		traditional_option = sample['mapping']['low_pole']
		traditional_option = sample['options'][traditional_option]
		secular_rational_option = sample['mapping']['high_pole']
		secular_rational_option = sample['options'][secular_rational_option]
		dataset.add_entry(f"{situation} You will {traditional_option}",f"{situation} You will {secular_rational_option}")
	model.reset()
	steering_vector = SteeringVector.train(model, dataset, method="mean_diff")
	if not test_data:
		test_data = train_data
	best_layers = find_best_layers(steering_vector, test_data, model, tokenizer)
 
	return steering_vector, best_layers
steering_vector, best_layers = get_steering_vector(train_data[:2], model, tokenizer)

Evaluating Scenarios: 100%|██████████| 2/2 [00:00<00:00, 15.20it/s]


Base score: 0.9954510629177094


Evaluating Scenarios: 100%|██████████| 2/2 [00:00<00:00, 14.62it/s]


In [22]:
for domain in ['Workplace', "Family", 'Legal']:
	train_data_X = filter(train_data, domain_list=[domain],id_list=X_axis_id)
	train_data_Y = filter(train_data, domain_list=[domain],id_list=Y_axis_id)
	print(f"{domain} - Train X: {len(train_data_X)}, Y: {len(train_data_Y)}")
	steering_vector_X, best_layers_X = get_steering_vector(train_data_X, model, tokenizer)
	steering_vector_Y, best_layers_Y = get_steering_vector(train_data_Y, model, tokenizer)
	print(f"  X-axis best layers: {best_layers_X[:10]}")
	print(f"  Y-axis best layers: {best_layers_Y[:10]}")
	best_layers_X = [i for i,_ in best_layers_X[:10]]
	best_layers_Y = [i for i,_ in best_layers_Y[:10]]
	# select top 4 common layers for both axes as final layers
	selected_layers = []
	for layer in best_layers_X:
		if layer in best_layers_Y:
			selected_layers.append(layer)
		if len(selected_layers) >= 4:
			break
	print(f"  Selected layers for both axes: {selected_layers}")
	selected_layers = [16, 17, 18, 19]
     
	# test the model with diffirent cofficients
	for x_cof in [-0.2, 0.2]:
		model.reset()
		model.layer_ids = selected_layers
		model.set_control(steering_vector_X, x_cof)
		results = evaluate_scenarios(sample_data, model, tokenizer)
		# save to json
		with open(f'./results_{domain}_X_{x_cof}.json', 'w') as f:
			json.dump(results, f, indent=2)
	
	for y_cof in [-0.2, 0.2]:
		model.reset()
		model.layer_ids = selected_layers
		model.set_control(steering_vector_Y, y_cof)
		results = evaluate_scenarios(sample_data, model, tokenizer)
		# save to json
		with open(f'./results_{domain}_Y_{y_cof}.json', 'w') as f:
			json.dump(results, f, indent=2)


Workplace - Train X: 50, Y: 50


Evaluating Scenarios: 100%|██████████| 50/50 [00:03<00:00, 13.74it/s]


Base score: 0.9239037007070147


Evaluating Scenarios: 100%|██████████| 50/50 [00:04<00:00, 11.88it/s]


Base score: 0.7302013593590345


...

Evaluating Scenarios: 100%|██████████| 50/50 [00:04<00:00, 12.09it/s]


Base score: 0.7968718658246507


Evaluating Scenarios: 100%|██████████| 50/50 [00:04<00:00, 12.27it/s]


  X-axis best layers: [(16, -0.025149738279633294), (17, -0.022552379471489914), (18, -0.012108571037351745), (14, -0.00794286718324655), (19, -0.007338770761289193), (13, -0.0072564465490359), (15, -0.0068461873560408115), (23, -0.003783651289013479), (12, -0.003581892383572205), (20, -0.003517755051270721)]
  Y-axis best layers: [(16, -0.024264142031788616), (17, -0.020644203094816516), (18, -0.011131154519753217), (19, -0.011012996975150569), (15, -0.008183599067197123), (20, -0.0051527181895653085), (13, -0.00392717756545613), (14, -0.0037576161133620056), (24, -0.0034633972540905056), (25, -0.0029535602426221663)]
  Selected layers for both axes: [16, 17, 18, 14]


Evaluating Scenarios: 100%|██████████| 300/300 [00:24<00:00, 12.19it/s]


In [23]:
for domain in ["All"]:
	train_data_X = filter(train_data,id_list=X_axis_id)
	train_data_Y = filter(train_data,id_list=Y_axis_id)
	print(f"{domain} - Train X: {len(train_data_X)}, Y: {len(train_data_Y)}")
	steering_vector_X, best_layers_X = get_steering_vector(train_data_X, model, tokenizer)
	steering_vector_Y, best_layers_Y = get_steering_vector(train_data_Y, model, tokenizer)
	print(f"  X-axis best layers: {best_layers_X[:10]}")
	print(f"  Y-axis best layers: {best_layers_Y[:10]}")
	best_layers_X = [i for i,_ in best_layers_X[:10]]
	best_layers_Y = [i for i,_ in best_layers_Y[:10]]
	# select top 4 common layers for both axes as final layers
	selected_layers = []
	for layer in best_layers_X:
		if layer in best_layers_Y:
			selected_layers.append(layer)
		if len(selected_layers) >= 4:
			break
	print(f"  Selected layers for both axes: {selected_layers}")
	selected_layers = [16, 17, 18, 19]
	# test the model with diffirent cofficients
	for x_cof in [-0.2, 0.2]:
		model.reset()
		model.layer_ids = selected_layers
		model.set_control(steering_vector_X, x_cof)
		results = evaluate_scenarios(sample_data, model, tokenizer)
		# save to json
		with open(f'./results_{domain}_X_{x_cof}.json', 'w') as f:
			json.dump(results, f, indent=2)
	
	for y_cof in [-0.2, 0.2]:
		model.reset()
		model.layer_ids = selected_layers
		model.set_control(steering_vector_Y, y_cof)
		results = evaluate_scenarios(sample_data, model, tokenizer)
		# save to json
		with open(f'./results_{domain}_Y_{y_cof}.json', 'w') as f:
			json.dump(results, f, indent=2)


All - Train X: 150, Y: 150


Evaluating Scenarios: 100%|██████████| 150/150 [00:12<00:00, 12.04it/s]


Base score: 0.8828811722136076


Evaluating Scenarios: 100%|██████████| 150/150 [00:12<00:00, 12.18it/s]


Base score: 0.7340929943352663


Evaluating Scenarios: 100%|██████████| 150/150 [00:12<00:00, 12.16it/s]


  X-axis best layers: [(16, -0.02439397242715735), (17, -0.020625535803828865), (18, -0.012587545578371717), (19, -0.007673844913236638), (15, -0.00688356253391853), (14, -0.00569221691306665), (13, -0.0047662417939015045), (12, -0.0034980455237625385), (20, -0.003145204228840237), (11, -0.0026574156882815947)]
  Y-axis best layers: [(17, -0.023407015310312973), (16, -0.022529503704681653), (18, -0.015809366767886468), (19, -0.014129887569943045), (20, -0.006596041029733879), (15, -0.00655152991856689), (14, -0.006223806041719682), (13, -0.003312935485688895), (12, -0.0029002599874790036), (11, -0.002891022711416169)]
  Selected layers for both axes: [16, 17, 18, 19]


Evaluating Scenarios: 100%|██████████| 300/300 [00:24<00:00, 12.33it/s]


In [24]:
!mkdir qwen_steering_outputs
!mv *.json qwen_steering_outputs/
!zip -r qwen_steering_outputs.zip qwen_steering_outputs/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  adding: qwen_steering_outputs/ (stored 0%)
  adding: qwen_steering_outputs/results_All_Y_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Workplace_Y_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_Family_X_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_Legal_X_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Family_Y_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_Workplace_X_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Workplace_Y_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Workplace_X_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_Legal_X_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_All_Y_-0.2.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 78%)
  adding: qwen_steering_outputs/results_Family_Y_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Family_X_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_All_X_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_Legal_Y_0.2.json (deflated 77%)
  adding: qwen_steering_outputs/results_Legal_Y_-0.2.json (deflated 78%)
  adding: qwen_steering_outputs/results_All_X_0.2.json (deflated 77%)


In [25]:
# top_layer_ids = [layer_id for layer_id, score in best_layers_Y[:4]]
# model.layer_ids = top_layer_ids
# for X_factor in [-0.2, 0.2]:
# 	for Y_factor in [-0.2, 0.2]:
# 		model.reset()
# 		combined_steering = steering_vector_X * X_factor + steering_vector_Y * Y_factor
# 		model.set_control(combined_steering, 1.0)
# 		scores = evaluate_scenarios(sample_data, model, tokenizer)
# 		print(f"X: {X_factor}, Y: {Y_factor} => ", get_normalized_scores(scores))
# 		answer_to_pivot(scores)

# 		with open(f'evaluation_results_X_{X_factor}_Y_{Y_factor}.json', 'w') as f:
# 			json.dump(scores, f, indent=2)	    
	


In [26]:
# X_factor = 0.3
# y_factor = -0.1
# combined_steering = steering_vector_X * X_factor + steering_vector_Y * y_factor
# print(f"\nSteering of x_factor {X_factor}, y_factor {y_factor} ")
# model.set_control(combined_steering, 1.0)

# print(ask_question("", "Someone suggests a 'flat hierarchy' at work. Do you support it answer in one sentence?", model, tokenizer))
# print(ask_question("", "You are deciding on a Sunday activity. Give me a short answer of What will you do?", model, tokenizer))


In [27]:
# X_factor = 0.3
# y_factor = 0.1
# combined_steering = steering_vector_X * X_factor + steering_vector_Y * y_factor
# print(f"\nSteering of x_factor {X_factor}, y_factor {y_factor} ")
# model.set_control(combined_steering, 1.0)

# print(ask_question("", "Someone suggests a 'flat hierarchy' at work. Do you support it answer in one sentence?", model, tokenizer))
# print(ask_question("", "You are deciding on a Sunday activity. Give me a short answer of What will you do?", model, tokenizer))


In [28]:
# example_Y = ["If you lose your job, you will take any available job", "If you lose your job, you will take time to retrain or find a role that aligns with your purpose"]

# # We can print all layers and identify which layer shows a clear distinction between the two outputs above.
# for layer in range(1,num_layers):
#     print(f"Layer {layer}: \t" + (visualize_activation(example_Y[0], model, steering_vector_Y, layer_index=layer) + " "
#         + visualize_activation(example_Y[1], model, steering_vector_Y, layer_index=layer)))

In [29]:
# from dialz.score import get_activation_score
# print(get_activation_score(example_Y[0], model, steering_vector_Y))
# print(get_activation_score(example_Y[1], model, steering_vector_Y))


In [30]:

# def create_default_profile_responses(model, tokenizer):
# 	profile_responses = {}
# 	for sample in tqdm(questions):
# 		answered_questions = []
# 		sample_question = sample['Question']
# 		for role in system_roles:
# 			for j in range(5):  
# 				thinking_content, content = ask_question(system_prompt_default_template.format(role=role), sample_question, model, tokenizer, thinking=False)
# 				answered_questions.append(content)
# 	profile_responses[sample['ID']] = answered_questions
# 	return profile_responses


# def create_profile_responses(model, tokenizer):
# 	profile_responses = {}
# 	for country in target_countries:
# 		print(f"Processing country: {country}")
# 		country_answers = {}
# 		for sample in tqdm(questions):
# 			answered_questions = []
# 			sample_question = sample['Question']
# 			for role in system_roles:
# 				for j in range(5):  
# 					thinking_content, content = ask_question(system_prompt_culture_align_template.format(role=role, country=country), sample_question, model, tokenizer, thinking=False)
# 					answered_questions.append(content)
# 			country_answers[sample['ID']] = answered_questions
# 		profile_responses[country] = country_answers
# 	return profile_responses

# def create_advance_profile_responses(model, tokenizer):
# 	profile_responses = {}
# 	for country in target_countries:
# 		print(f"Processing country: {country}")
# 		country_answers = {}
# 		advance_steering_prompt = advance_steering_prompts[country]
# 		for sample in tqdm(questions):
# 			answered_questions = []
# 			sample_question = sample['Question']
# 			for j in range(25):  
# 				thinking_content, content = ask_question(advance_steering_prompt, sample_question, model, tokenizer, thinking=False)
# 				answered_questions.append(content)
# 			country_answers[sample['ID']] = answered_questions
# 		profile_responses[country] = country_answers
# 	return profile_responses

# # for model_name in ["Qwen/Qwen3-4B","meta-llama/Llama-3.2-3B-Instruct"]:
# # 	print(f"Processing model: {model_name}")
# # 	# load the tokenizer and the model
# # 	tokenizer = AutoTokenizer.from_pretrained(model_name)
# # 	model = AutoModelForCausalLM.from_pretrained(
# # 	    model_name,
# # 	    torch_dtype="auto",
# # 	    device_map="auto"
# # 	)

# model.set_control(combined_steering, 0)

# default_profile = create_default_profile_responses(model, tokenizer)
# with open(f'default_profile_responses_{model_name.replace("/", "_")}.json', 'w') as f:
# 	json.dump(default_profile, f)

# profile_responses = create_profile_responses(model, tokenizer)
# with open(f'prompt_basic_steering_{model_name.replace("/", "_")}.json', 'w') as f:
#     json.dump(profile_responses, f)
    
# profile_responses = create_advance_profile_responses(model, tokenizer)
# with open(f'prompt_advance_steering_{model_name.replace("/", "_")}.json', 'w') as f:
#     json.dump(profile_responses, f)

# model.set_control(steering_vector_X, 0.5)
# profile_responses = create_profile_responses(model, tokenizer)
# with open(f'vector_and_basic_steering_0.5X_{model_name.replace("/", "_")}.json', 'w') as f:
#     json.dump(profile_responses, f)

# profile_responses = create_default_profile_responses(model, tokenizer)
# with open(f'vector_steering_0.5X_{model_name.replace("/", "_")}.json', 'w') as f:
#     json.dump(profile_responses, f)


# model.set_control(steering_vector_Y, 0.5)
# profile_responses = create_profile_responses(model, tokenizer)
# with open(f'vector_and_basic_steering_0.5Y_{model_name.replace("/", "_")}.json', 'w') as f:
#     json.dump(profile_responses, f)
# profile_responses = create_default_profile_responses(model, tokenizer)
# with open(f'vector_steering_0.5Y_{model_name.replace("/", "_")}.json', 'w') as f:
# 	json.dump(profile_responses, f)